## Nombre: Gustavo Hernández Angeles

### Leer datos

In [2]:
import os
import re
from nltk import FreqDist
def get_texts_from_file(path_corpus, path_truth):
    tr_txt = []
    tr_y = []

    # Manera más chida de abrir (y cerrar auto) archivos.
    with open(path_corpus, "r", encoding="utf-8-sig") as f_corpus, open(path_truth, "r", encoding="utf-8-sig") as f_truth:
        for twitt in f_corpus:
            tr_txt += [twitt]
        for label in f_truth:
            tr_y += [label]

    return tr_txt, tr_y

In [3]:
tr_txt, tr_y = get_texts_from_file("./data/mex_train.txt","./data/mex_train_labels.txt")

### Preprocesamiento y tratamiento de datos

In [4]:
class TrigramData:
    def __init__(self, vocab_max, tokenizer):
        self.vocab_max = vocab_max
        self.tokenizer = tokenizer
        self.UNK = "<UNK>"
        self.SOS = "<s>"
        self.EOS = "</s>"
        self.final_vocabulary = set()
    
    def fit(self, raw_texts):
        freqdist = FreqDist()
        tokenized_corpus = []
        
        for txt in raw_texts:
            tokens = self.tokenizer.tokenize(txt)
            tokenized_corpus.append(tokens)
            for w in tokens:
                freqdist[w] += 1
                
        self.final_vocabulary = {tok for tok, _ in freqdist.most_common(self.vocab_max)}
        self.final_vocabulary.update([self.UNK, self.SOS, self.EOS])
        
        transformed_corpus = []
        for tokens in tokenized_corpus:
            transformed_corpus.append(self.transform(tokens))
        return transformed_corpus
    
    def mask_oov(self, w):
        return self.UNK if w not in self.final_vocabulary else w
    
    def add_sos_eos(self, tokens):
        return [self.SOS, self.SOS] + tokens + [self.EOS]
    
    def transform(self, tokens):
        transformed = []
        for w in tokens:
            transformed.append(self.mask_oov(w))
        transformed = self.add_sos_eos(transformed)
        return transformed

### TrigramLM

In [5]:
class TrigramLanguageModel:
    def __init__(self, lambda1 = 0.4, lambda2 = 0.3, lambda3=0.3):
        self.lambda1 = lambda1
        self.lambda2 = lambda2
        self.lambda3 = lambda3
        
        # Contadores
        self.trigram_freq = {}
        self.bigram_freq = {}
        self.unigram_freq = {}
        
        self.total_tokens = 0
        self.vocabulary = 0
        self.V = 0
    
    def train(self, transformed_corpus, final_vocabulary):
        self.vocabulary = final_vocabulary
        self.V = len(final_vocabulary)
        
        for tokens in transformed_corpus:
            for i, w in enumerate(tokens):
                
                # Unigrama
                self.unigram_freq[w] = self.unigram_freq.get(w, 0) + 1
                
                # Bigrama
                if i > 0:
                    w_prev = tokens[i-1]
                    self.bigram_freq[(w_prev,w)] = self.bigram_freq.get((w_prev,w), 0) + 1
                
                # Trigrama
                if i > 1:
                    w_prev_prev = tokens[i-2]
                    w_prev = tokens[i-1]
                    self.trigram_freq[(w_prev_prev,w_prev,w)] = \
                        self.trigram_freq.get((w_prev_prev,w_prev,w),0) + 1
                
                self.total_tokens = sum(self.unigram_freq.values())
        
    def mask_oov(self, w):
        return "<UNK>" if w not in self.vocabulary else w
    
    def unigram_probability(self, w):
        # Add-one smoothing
        return (self.unigram_freq.get(self.mask_oov(w), 0)+1)/(self.total_tokens+self.V)
    
    def bigram_probability(self, w_prev, w):
        # Add-one smoothing
        w_prev = self.mask_oov(w_prev)
        w = self.mask_oov(w)
        
        numerator = self.bigram_freq.get((w_prev,w), 0) + 1
        denominator = self.unigram_freq.get(w_prev, 0) + self.V
        return numerator/denominator
    
    def trigram_probability(self, w_prev2, w_prev, w):
        # Add-one smoothing
        w_prev2 = self.mask_oov(w_prev2)
        w_prev = self.mask_oov(w_prev)
        w = self.mask_oov(w)
        
        numerator = self.trigram_freq.get((w_prev2,w_prev,w), 0) + 1
        denominator = self.bigram_freq.get((w_prev, w), 0) + self.V
        return numerator/denominator
    
    def probability_of_sentence(self, w_prev2,w_prev,w):
        return self.lambda3*self.unigram_probability(w) + \
               self.lambda2*self.bigram_probability(w_prev,w) + \
               self.lambda1*self.trigram_probability(w_prev2,w_prev,w)
               
    def sequence_probability(self, sequence):
        import math
        log_prob = 0
        for i in range(2, len(sequence)):
            w_prev2 = sequence[i-2]
            w_prev = sequence[i-1]
            w = sequence[i]
            p = self.probability_of_sentence(w_prev2, w_prev, w)
            
            log_prob += math.log(p)
            prob += p
        return math.exp(log_prob)
    
    def checar_probas(self):
        print(sum(self.unigram_probability(w) for w in self.vocabulary))
        print(sum(self.bigram_probability("hola", w) for w in self.vocabulary))
        print(sum(self.trigram_probability("hola", "como", w) for w in self.vocabulary))

In [6]:
from nltk.tokenize import TweetTokenizer

tokenizer = TweetTokenizer()

trigram_data = TrigramData(vocab_max=15194, tokenizer=tokenizer)
transformed_corpus = trigram_data.fit(tr_txt)
final_vocab = trigram_data.final_vocabulary

In [7]:
trigram_lm = TrigramLanguageModel(lambda1=0.6, lambda2 = 0.3, lambda3 = 0.1)

In [8]:
trigram_lm.train(transformed_corpus, final_vocab)

In [9]:
trigram_lm.checar_probas()

1.0
1.0
0.9999981998457141


### Pruebitas

In [10]:
w_prev2, w_prev, w = "<s>", "hola", "mundo"
p_w = trigram_lm.probability_of_sentence(w_prev2, w_prev, w)
print(f"P({w}|{w_prev2},{w_prev}) = {p_w}")

P(mundo|<s>,hola) = 0.00010123776032573012


In [ ]:
w_prev2, w_prev, w = "hola", "pinche", "pendejo"
p_w = trigram_lm.probability_of_sentence(w_prev2, w_prev, w)
print(f"P({w}|{w_prev2},{w_prev}) = {p_w}")

P(pendejo|hola,pinche) = 0.00011504211090654653
